# Vattenfall Silver Layer - Comprehensive Data Validation

## Purpose
Validate all Silver layer tables before building Gold layer aggregations.

## Silver Tables to Validate
1. **silver_grid_events** - Core operational events
2. **silver_asset_reference** - Asset master data
3. **silver_regional_operations_base** - Integrated operational dataset (optional)
4. **silver_market_prices** - Market data
5. **silver_weather** - Weather observations

## Validation Checks
* ✅ Table existence and row counts
* ✅ Key column nulls
* ✅ Invalid values and outliers
* ✅ Schema consistency
* ✅ Referential integrity
* ✅ Data quality scores

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from typing import Dict, List, Tuple
import sys

# Add validation module to path
sys.path.append('/Workspace/Users/gharbi@startsteps.org/vattenfall-capstone-project/src')

print("✅ Imports complete")
print("✅ Validation framework ready")

In [0]:
def check_table_exists(table_name: str) -> Dict:
    """
    Check if table exists and return basic metrics.
    """
    try:
        df = spark.table(table_name)
        row_count = df.count()
        col_count = len(df.columns)
        
        return {
            "exists": True,
            "table_name": table_name,
            "row_count": row_count,
            "column_count": col_count,
            "status": "OK" if row_count > 0 else "EMPTY"
        }
    except Exception as e:
        return {
            "exists": False,
            "table_name": table_name,
            "error": str(e),
            "status": "MISSING"
        }

print("✅ Helper function loaded: check_table_exists")

In [0]:
def check_nulls(df: DataFrame, key_columns: List[str]) -> Dict:
    """
    Check for null values in key columns.
    """
    total_rows = df.count()
    null_results = {}
    
    for col in key_columns:
        if col in df.columns:
            null_count = df.filter(F.col(col).isNull()).count()
            null_pct = (null_count / total_rows * 100) if total_rows > 0 else 0
            
            null_results[col] = {
                "null_count": null_count,
                "null_percentage": round(null_pct, 2),
                "status": "FAIL" if null_count > 0 else "PASS"
            }
        else:
            null_results[col] = {"status": "COLUMN_NOT_FOUND"}
    
    return null_results

print("✅ Helper function loaded: check_nulls")

In [0]:
def check_value_ranges(df: DataFrame, range_checks: Dict[str, Tuple]) -> Dict:
    """
    Check if values are within expected ranges.
    
    Args:
        range_checks: Dict of {column: (min_val, max_val, nullable)}
    """
    results = {}
    total_rows = df.count()
    
    for col, (min_val, max_val, nullable) in range_checks.items():
        if col not in df.columns:
            results[col] = {"status": "COLUMN_NOT_FOUND"}
            continue
        
        # Count out-of-range values (excluding nulls if nullable)
        if nullable:
            invalid_count = df.filter(
                F.col(col).isNotNull() & 
                ((F.col(col) < min_val) | (F.col(col) > max_val))
            ).count()
        else:
            invalid_count = df.filter(
                (F.col(col).isNull()) | 
                (F.col(col) < min_val) | 
                (F.col(col) > max_val)
            ).count()
        
        invalid_pct = (invalid_count / total_rows * 100) if total_rows > 0 else 0
        
        results[col] = {
            "expected_range": f"[{min_val}, {max_val}]",
            "invalid_count": invalid_count,
            "invalid_percentage": round(invalid_pct, 2),
            "nullable": nullable,
            "status": "FAIL" if invalid_count > 0 else "PASS"
        }
    
    return results

print("✅ Helper function loaded: check_value_ranges")

In [0]:
print("="*70)
print("MILESTONE 1: VALIDATE SILVER FOUNDATION")
print("="*70)

# Define SILVER tables to check
tables_to_check = [
    "vattenfall_dev.refined.silver_grid_events",
    "vattenfall_dev.refined.silver_asset_reference",
    "vattenfall_dev.refined.silver_regional_operations_base",
    "vattenfall_dev.refined.silver_market_prices",
    "vattenfall_dev.refined.silver_weather"
]

print("\n1. TABLE EXISTENCE & ROW COUNTS")
print("-" * 70)

table_summary = []
for table in tables_to_check:
    result = check_table_exists(table)
    table_summary.append(result)
    
    status_icon = "✅" if result["status"] == "OK" else "❌"
    if result["exists"]:
        print(f"{status_icon} {table}")
        print(f"   Rows: {result['row_count']:,} | Columns: {result['column_count']}")
    else:
        print(f"{status_icon} {table} - MISSING")
        print(f"   Error: {result.get('error', 'Unknown')}")

# Summary
total_tables = len(tables_to_check)
valid_tables = sum(1 for t in table_summary if t["status"] == "OK")

print(f"\n{'='*70}")
print(f"Summary: {valid_tables}/{total_tables} Silver tables exist and have data")
print(f"{'='*70}\n")

In [0]:
print("\n2. VALIDATE: silver_grid_events")
print("-" * 70)

# Load table
df_events = spark.table("vattenfall_dev.refined.silver_grid_events")

print(f"\nLoaded: {df_events.count():,} events")
print(f"Total columns: {len(df_events.columns)}\n")

# Check ALL columns for nulls
print("Column Null Check (ALL COLUMNS):")
null_results = check_nulls(df_events, df_events.columns)
failed_cols = []
for col, result in null_results.items():
    if result.get("status") in ["PASS", "FAIL"]:
        if result.get("status") == "FAIL":
            status_icon = "❌"
            failed_cols.append(col)
            print(f"  {status_icon} {col}: {result['null_count']} nulls ({result['null_percentage']}%)")
    elif result.get("status") == "COLUMN_NOT_FOUND":
        print(f"  ⚠️ {col}: {result['status']}")

if not failed_cols:
    print("  ✅ All columns have zero nulls")

# Check value ranges
print("\nValue Range Checks:")
range_checks = {
    "duration_minutes": (0, 1440, True),  # 0-24 hours, nullable
    "affected_customers": (0, 1000000, True),  # Up to 1M customers, nullable
    "data_quality_score": (0, 100, False),  # Required 0-100
}
range_results = check_value_ranges(df_events, range_checks)
for col, result in range_results.items():
    status_icon = "✅" if result.get("status") == "PASS" else "❌"
    if result.get("status") in ["PASS", "FAIL"]:
        print(f"  {status_icon} {col}: {result['invalid_count']} invalid ({result['invalid_percentage']}%)")
    else:
        print(f"  ⚠️ {col}: {result['status']}")

# Show severity distribution
print("\nSeverity Distribution:")
df_events.groupBy("severity").count().orderBy(F.desc("count")).show()

print("✅ silver_grid_events validation complete\n")

In [0]:
print("\n3. VALIDATE: silver_asset_reference")
print("-" * 70)

# Load table
df_assets = spark.table("vattenfall_dev.refined.silver_asset_reference")

print(f"\nLoaded: {df_assets.count():,} assets")
print(f"Total columns: {len(df_assets.columns)}\n")

# Check ALL columns for nulls
print("Column Null Check (ALL COLUMNS):")
null_results = check_nulls(df_assets, df_assets.columns)
failed_cols = []
for col, result in null_results.items():
    if result.get("status") in ["PASS", "FAIL"]:
        if result.get("status") == "FAIL":
            status_icon = "❌"
            failed_cols.append(col)
            print(f"  {status_icon} {col}: {result['null_count']} nulls ({result['null_percentage']}%)")
    elif result.get("status") == "COLUMN_NOT_FOUND":
        print(f"  ⚠️ {col}: {result['status']}")

if not failed_cols:
    print("  ✅ All columns have zero nulls")

# Check value ranges
print("\nValue Range Checks:")
range_checks = {
    "capacity_mva": (0, 2000, False),  # Up to 2000 MVA capacity
    "asset_age_years": (0, 100, False),  # 0-100 years
    "region_population": (0, 10000000, False),  # Up to 10M population
    "region_area_km2": (0, 500000, False),  # Up to 500k km²
}
range_results = check_value_ranges(df_assets, range_checks)
for col, result in range_results.items():
    status_icon = "✅" if result.get("status") == "PASS" else "❌"
    if result.get("status") in ["PASS", "FAIL"]:
        print(f"  {status_icon} {col}: {result['invalid_count']} invalid ({result['invalid_percentage']}%)")
    else:
        print(f"  ⚠️ {col}: {result['status']}")

# Show asset age distribution
print("\nAsset Age Category Distribution:")
df_assets.groupBy("asset_age_category").count().orderBy("asset_age_category").show()

print("✅ silver_asset_reference validation complete\n")

In [0]:
print("\n4. VALIDATE: silver_regional_operations_base")
print("-" * 70)

# Load table
df_regional = spark.table("vattenfall_dev.refined.silver_regional_operations_base")

print(f"\nLoaded: {df_regional.count():,} integrated records")
print(f"Total columns: {len(df_regional.columns)}\n")

# Check ALL columns for nulls
print("Column Null Check (ALL COLUMNS):")
null_results = check_nulls(df_regional, df_regional.columns)
failed_cols = []
for col, result in null_results.items():
    if result.get("status") in ["PASS", "FAIL"]:
        if result.get("status") == "FAIL":
            status_icon = "❌"
            failed_cols.append(col)
            print(f"  {status_icon} {col}: {result['null_count']} nulls ({result['null_percentage']}%)")
    elif result.get("status") == "COLUMN_NOT_FOUND":
        print(f"  ⚠️ {col}: {result['status']}")

if not failed_cols:
    print("  ✅ All columns have zero nulls")

# Show high-risk event count
print("\nHigh-Risk Events:")
high_risk_count = df_regional.filter(F.col("high_risk_event") == True).count()
print(f"  Total high-risk events: {high_risk_count}")

print("✅ silver_regional_operations_base validation complete\n")

In [0]:
print("\n5. VALIDATE: silver_market_prices")
print("-" * 70)

# Load table
df_prices = spark.table("vattenfall_dev.refined.silver_market_prices")

print(f"\nLoaded: {df_prices.count():,} price observations")
print(f"Total columns: {len(df_prices.columns)}\n")

# Check ALL columns for nulls
print("Column Null Check (ALL COLUMNS):")
null_results = check_nulls(df_prices, df_prices.columns)
failed_cols = []
for col, result in null_results.items():
    if result.get("status") in ["PASS", "FAIL"]:
        if result.get("status") == "FAIL":
            status_icon = "❌"
            failed_cols.append(col)
            print(f"  {status_icon} {col}: {result['null_count']} nulls ({result['null_percentage']}%)")
    elif result.get("status") == "COLUMN_NOT_FOUND":
        print(f"  ⚠️ {col}: {result['status']}")

if not failed_cols:
    print("  ✅ All columns have zero nulls")

# Check value ranges
print("\nValue Range Checks:")
range_checks = {
    "price_eur_mwh": (0, 500, False),  # Reasonable market price range
}
range_results = check_value_ranges(df_prices, range_checks)
for col, result in range_results.items():
    status_icon = "✅" if result.get("status") == "PASS" else "❌"
    if result.get("status") in ["PASS", "FAIL"]:
        print(f"  {status_icon} {col}: {result['invalid_count']} invalid ({result['invalid_percentage']}%)")
    else:
        print(f"  ⚠️ {col}: {result['status']}")

# Show price statistics
print("\nPrice Statistics:")
df_prices.select(
    F.min("price_eur_mwh").alias("min_price"),
    F.avg("price_eur_mwh").alias("avg_price"),
    F.max("price_eur_mwh").alias("max_price")
).show()

print("✅ silver_market_prices validation complete\n")

In [0]:
print("\n6. VALIDATE: silver_weather")
print("-" * 70)

# Load table
df_weather = spark.table("vattenfall_dev.refined.silver_weather")

print(f"\nLoaded: {df_weather.count():,} weather observations")
print(f"Total columns: {len(df_weather.columns)}\n")

# Check ALL columns for nulls
print("Column Null Check (ALL COLUMNS):")
null_results = check_nulls(df_weather, df_weather.columns)
failed_cols = []
for col, result in null_results.items():
    if result.get("status") in ["PASS", "FAIL"]:
        if result.get("status") == "FAIL":
            status_icon = "❌"
            failed_cols.append(col)
            print(f"  {status_icon} {col}: {result['null_count']} nulls ({result['null_percentage']}%)")
    elif result.get("status") == "COLUMN_NOT_FOUND":
        print(f"  ⚠️ {col}: {result['status']}")

if not failed_cols:
    print("  ✅ All columns have zero nulls")

# Check value ranges
print("\nValue Range Checks:")
range_checks = {
    "temperature_c": (-50, 60, False),  # Nordic climate range
    "wind_speed_ms": (0, 100, True),  # Wind speed in m/s, nullable
    "precipitation_mm": (0, 500, True),  # Precipitation, nullable
    "cloud_cover_percent": (0, 100, True),  # Cloud cover, nullable
}
range_results = check_value_ranges(df_weather, range_checks)
for col, result in range_results.items():
    status_icon = "✅" if result.get("status") == "PASS" else "❌"
    if result.get("status") in ["PASS", "FAIL"]:
        print(f"  {status_icon} {col}: {result['invalid_count']} invalid ({result['invalid_percentage']}%)")
    else:
        print(f"  ⚠️ {col}: {result['status']}")

# Show weather statistics
print("\nWeather Statistics:")
df_weather.select(
    F.min("temperature_c").alias("min_temp"),
    F.avg("temperature_c").alias("avg_temp"),
    F.max("temperature_c").alias("max_temp")
).show()

print("✅ silver_weather validation complete\n")

In [0]:
print("\n7. REFERENTIAL INTEGRITY CHECKS")
print("-" * 70)

# Check 1: Events → Assets
print("\nCheck 1: Events → Assets (via substation_id)")
events_with_assets = df_events.join(
    df_assets.select("substation_id").distinct(),
    "substation_id",
    "inner"
).count()

total_events = df_events.count()
orphaned_events = total_events - events_with_assets
integrity_pct = (events_with_assets / total_events * 100) if total_events > 0 else 0

status_icon = "✅" if orphaned_events == 0 else "⚠️"
print(f"  {status_icon} Matched events: {events_with_assets:,} / {total_events:,} ({integrity_pct:.1f}%)")
if orphaned_events > 0:
    print(f"  ⚠️ Orphaned events (no matching asset): {orphaned_events:,}")

# Check 2: Regional Operations → Events
print("\nCheck 2: Regional Operations → Events (via event_id)")
regional_with_events = df_regional.join(
    df_events.select("event_id").distinct(),
    "event_id",
    "inner"
).count()

total_regional = df_regional.count()
integrity_pct = (regional_with_events / total_regional * 100) if total_regional > 0 else 0

status_icon = "✅" if integrity_pct == 100 else "❌"
print(f"  {status_icon} Matched records: {regional_with_events:,} / {total_regional:,} ({integrity_pct:.1f}%)")

# Check 3: Regional Operations → Assets
print("\nCheck 3: Regional Operations → Assets (via substation_id)")
regional_with_assets = df_regional.join(
    df_assets.select("substation_id").distinct(),
    "substation_id",
    "inner"
).count()

integrity_pct = (regional_with_assets / total_regional * 100) if total_regional > 0 else 0

status_icon = "✅" if integrity_pct == 100 else "❌"
print(f"  {status_icon} Matched records: {regional_with_assets:,} / {total_regional:,} ({integrity_pct:.1f}%)")

print("\n✅ Referential integrity checks complete\n")

In [0]:
print("\n8. SCHEMA CONSISTENCY CHECKS")
print("-" * 70)

# Check that expected columns exist in each table
print("\nExpected Columns Check:")

expected_schemas = {
    "silver_grid_events": [
        "event_id", "timestamp", "substation_id", "event_type", 
        "severity", "duration_minutes", "affected_customers", "total_impact_score", 
        "data_quality_score"
    ],
    "silver_asset_reference": [
        "asset_key", "substation_id", "substation_name", "region_name", 
        "capacity_mva", "voltage_level", "asset_age_years", "asset_age_category",
        "region_population", "region_area_km2"
    ],
    "silver_regional_operations_base": [
        "event_id", "substation_id", "asset_key", "region_key", "country_name",
        "impact_intensity", "duration_severity_score", "high_risk_event",
        "population_impact_rate", "regional_event_density"
    ]
}

tables_map = {
    "silver_grid_events": df_events,
    "silver_asset_reference": df_assets,
    "silver_regional_operations_base": df_regional
}

for table_name, expected_cols in expected_schemas.items():
    df = tables_map[table_name]
    actual_cols = set(df.columns)
    expected_cols_set = set(expected_cols)
    
    missing_cols = expected_cols_set - actual_cols
    extra_cols = actual_cols - expected_cols_set
    
    print(f"\n{table_name}:")
    if len(missing_cols) == 0:
        print(f"  ✅ All expected columns present ({len(expected_cols)} columns)")
    else:
        print(f"  ❌ Missing columns: {missing_cols}")
    
    if len(extra_cols) > 0:
        print(f"  ℹ️ Additional columns: {len(extra_cols)} (enrichments/audit fields)")

print("\n✅ Schema consistency checks complete\n")

In [0]:
print("="*70)
print("SILVER LAYER VALIDATION - FINAL SUMMARY")
print("="*70)

print("\n✅ VALIDATION COMPLETE\n")

print("Table Status:")
print(f"  ✅ silver_grid_events: {df_events.count():,} records, {len(df_events.columns)} columns")
print(f"  ✅ silver_asset_reference: {df_assets.count():,} records, {len(df_assets.columns)} columns")
print(f"  ✅ silver_regional_operations_base: {df_regional.count():,} records, {len(df_regional.columns)} columns")
print(f"  ✅ silver_market_prices: {df_prices.count():,} records, {len(df_prices.columns)} columns")
print(f"  ✅ silver_weather: {df_weather.count():,} records, {len(df_weather.columns)} columns")

print("\n⚠️  Key Findings:")

# Calculate orphaned events
orphaned = df_events.count() - df_events.join(df_assets.select("substation_id").distinct(), "substation_id", "inner").count()
if orphaned > 0:
    print(f"  ⚠️ {orphaned:,} events without matching assets (referential integrity issue)")
else:
    print("  ✅ All events have matching assets")

# Check for high-risk events
high_risk = df_regional.filter(F.col("high_risk_event") == True).count()
if high_risk > 0:
    print(f"  ⚠️ {high_risk} high-risk events identified (aging assets + critical severity)")

print("\n✅ READY FOR GOLD LAYER DEVELOPMENT")
print("\nNext Steps:")
print("  1. Address orphaned events (if any)")
print("  2. Build Gold layer aggregation tables")
print("  3. Create executive dashboards")
print("  4. Implement real-time monitoring")

print("\n" + "="*70)
print("✅ VALIDATION REPORT COMPLETE")
print("="*70)